# Government Bond Yields: US, Japan, Germany, UK and Australia

Daily 10-year and 30-year benchmark government bond yields for five major markets.

**Sources**

| Country | Source | 10-year from | 30-year from |
|---|---|---|---|
| United States | Yahoo Finance `^TNX` / `^TYX` constant-maturity indices | 1962 | 1977 |
| Japan | Ministry of Finance daily JGB curve (CSV) | Jul 1986 | Sep 1999 |
| Germany | Bundesbank daily term structure of listed Federal securities | Aug 1997 | Aug 2000 |
| United Kingdom | Bank of England fitted nominal gilt curve | 1979 | **2016** |
| Australia | RBA table F2 (daily), spliced to the pre-2013 historical table | 1995 | **none published** |

Each line begins when its own data begins; no series is truncated to a common
window. The charts themselves start in 1986 (10-year) and 1999 (30-year).

**Caveats**

- **Australia appears at ten years only.** The RBA's capital market yields stop
  at a 10-year maturity, and the AOFM publishes no daily secondary-market yield
  curve, so there is no published Australian 30-year constant-maturity yield to
  plot. Australia issued its first 30-year bond only in 2016.
- **These are different constructs.** US yields are constant-maturity
  (par-equivalent), Japan's are yields on benchmark bonds, Germany's and the
  UK's are fitted zero-coupon spot rates, and Australia's are the RBA's
  interpolated bond yields. At ten years the differences are minor; at thirty
  years spot and par yields can diverge noticeably. The comparison is the right
  one to make, but the series are not computed identically.
- **UK 30-year only reaches back to 2016.** The BoE's pre-2016 workbooks stop at
  a 25-year maturity, so there is no 30-year gilt yield to plot before then.
- **US 30-year, Feb 2002 to Feb 2006.** Treasury issued no 30-year bond over that
  stretch. FRED's `DGS30` has a genuine four-year hole; `^TYX` does not, because
  CBOE kept quoting off the longest bond still outstanding, whose maturity was
  drifting down toward 25 years. The 30-year chart carries this as a header note.
- Publication lags differ by a few days, so the lines need not end on the same
  date.

**Downloading and caching**

Everything is cached in the project's shared `./CACHE`, under the usual
`prefix--hash--filename` convention.

- The **MOF** files go through `readabs.download_cache.get_file()`: they send
  `Last-Modified`, so it re-downloads only on a new publication and falls back
  to the cached copy if a download fails.
- The **Bundesbank** and **Bank of England** cannot use `get_file()`, for two
  different reasons, so they share `fetch_cached()` in the next cell. The BoE
  answers a request without a User-Agent with a 403, and `get_file()` cannot be
  given headers. The Bundesbank sends no `Last-Modified` at all, so `get_file()`
  would serve its first download forever and the German line would silently stop
  updating. `fetch_cached()` downloads fresh, caches, and falls back to the
  cached copy when a download fails - which matters because the Bundesbank
  throttles bursts of requests behind a proof-of-work challenge, and without the
  fallback a throttle takes the whole notebook down.
- The BoE history archive is about 39 MB and only changes as years roll over, so
  it alone is fetched conditionally on the server's `Last-Modified`.

**Other source quirks:** the MOF files served from the `/english/` path still
carry Shift-JIS bytes, so they are decoded as `cp932`; the Bundesbank CSVs put a
variable number of metadata rows ahead of the data; and the BoE's current
workbook carries a literal "Refresh" row, which leaves the yield columns as
object dtype unless they are coerced - an object column plots but silently loses
its end-point label.

## Setup

In [1]:
# system imports
import io
import re
import zipfile
from datetime import UTC, datetime
from functools import cache
from hashlib import sha256
from pathlib import Path
from typing import cast

# analytic imports
import pandas as pd
import readabs as ra
import requests
import yfinance as yf
from readabs.download_cache import (
    BAD_CACHE_PATTERN,
    get_file,
    retrieve_from_cache,
    save_to_cache,
)

# local imports
import mgplot as mg

In [2]:
# pandas display
pd.options.display.max_rows = 999999

# chart output directory
CHART_DIR = "./CHARTS/Bonds/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

# display charts inline?
SHOW = False

# Chart windows: full history, and roughly the last three years of trading days.
RECENT_TRADING_DAYS = -750
plot_times = 0, RECENT_TRADING_DAYS

# Series code per tenor and country: a Yahoo ticker (US), a MOF curve column
# (Japan), a Bundesbank series key (Germany), a BoE curve maturity in years
# (UK), or an RBA series title (Australia). Countries start when their data
# starts; no series is truncated to a common window. Australia appears only at
# ten years - neither the RBA nor the AOFM publishes a 30-year yield.
CODES: dict[str, dict[str, str]] = {
    "10-year": {
        "United States": "^TNX",
        "Japan": "10Y",
        "Germany": "D.I.ZST.ZI.EUR.S1311.B.A604.R10XX.R.A.A._Z._Z.A",
        "United Kingdom": "10",
        "Australia": "Australian Government 10 year bond",
    },
    "30-year": {
        "United States": "^TYX",
        "Japan": "30Y",
        "Germany": "D.I.ZST.ZI.EUR.S1311.B.A604.R30XX.R.A.A._Z._Z.A",
        "United Kingdom": "30",
    },
}

# Shorter names for chart titles; anything absent is used as it stands.
SHORT_NAMES: dict[str, str] = {
    "United States": "US",
    "United Kingdom": "UK",
}

# Two-letter codes for the footer, which names every series at once.
ABBREVIATIONS: dict[str, str] = {
    "United States": "US",
    "Japan": "JP",
    "Germany": "DE",
    "United Kingdom": "UK",
    "Australia": "AU",
}

# Where each chart begins. The 10-year window opens when Japan joins in 1986;
# the 30-year when the JGB 30-year sector opened in 1999.
TENOR_STARTS: dict[str, str] = {
    "10-year": "1986-01-01",
    "30-year": "1999-01-01",
}

# Comparability caveats, shown as that chart's rheader.
TENOR_NOTES: dict[str, str | None] = {
    "10-year": None,
    "30-year": (
        "US Feb 2002 to Feb 2006: no 30-year issuance, so ^TYX tracks the "
        "longest bond outstanding. UK curve reaches 30 years only from 2016"
    ),
}

# The earliest date asked of Yahoo Finance; each chart's own start does the
# trimming.
EARLIEST_START = "1960-01-01"

# The project's shared download cache, and a prefix per source.
CACHE_DIR = Path("./CACHE")
MOF_CACHE_PREFIX = "jgb_curve"
BBK_CACHE_PREFIX = "bund_yields"
BOE_CACHE_PREFIX = "boe_yield_curve"

# The Bank of England answers a header-less request with a 403.
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
}
HEAD_TIMEOUT = 30
DOWNLOAD_TIMEOUT = 300

# Japan MOF daily JGB yield curve: a full history file plus the current month.
MOF_BASE = "https://www.mof.go.jp/english/policy/jgbs/reference/interest_rate/"
MOF_HISTORY = "historical/jgbcme_all.csv"
MOF_CURRENT = "jgbcme.csv"
MOF_ENCODING = "cp932"  # the "English" CSVs still contain Shift-JIS bytes

# Bundesbank daily term structure of listed Federal securities (Svensson).
BBK_BASE = "https://api.statistiken.bundesbank.de/rest/download/BBSIS/"

# Bank of England fitted nominal gilt curve. The history archive is ~39 MB, so
# it is only re-downloaded when the server reports a newer copy; the
# current-month archive is small and fetched every run.
BOE_BASE = "https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/"
BOE_HISTORY = "glcnominalddata.zip"
BOE_CURRENT = "latest-yield-curve-data.zip"

# RBA daily capital market yields, split into a current and a pre-2013 table.
RBA_CURRENT_TABLE = "F2"
RBA_HISTORY_TABLE = "Z:F2-Daily-2013"

# Metadata rows precede the data in the MOF and Bundesbank files, and the count
# is not constant, so data rows are recognised by their leading date.
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2},")

# footer source attribution
SOURCE = (
    "Sources: Yahoo Finance; Japan MOF; Bundesbank; Bank of England; RBA"
)

## Data capture

In [3]:
def cache_path(url: str, prefix: str) -> Path:
    """The readabs cache file name for a URL, so these downloads sit in ./CACHE
    under the same prefix--hash--filename convention as everything else."""
    digest = sha256(url.encode("utf-8")).hexdigest()
    tail = url.rsplit("/", 1)[-1].split("?", 1)[0]
    return CACHE_DIR / re.sub(BAD_CACHE_PATTERN, "", f"{prefix}--{digest}--{tail}")


def fetch_cached(url: str, prefix: str, *, conditional: bool = False) -> bytes:
    """Download a URL that readabs' own fetchers cannot handle, and cache it.

    `get_file()` is the right tool wherever it works, and the MOF files use it.
    It is wrong for the other two sources here:

    - The Bank of England answers a request without a User-Agent with a 403,
      and neither `get_file()` nor `request_get()` can be given headers.
    - The Bundesbank sends no Last-Modified header. With nothing to compare
      against, `get_file()` returns its first download forever, so the German
      series would silently stop updating.

    So this downloads afresh, caches the bytes, and falls back to the cached
    copy when the download fails - which is what stops a transient block (the
    Bundesbank throttles bursts of requests) from failing the whole notebook.
    `conditional` skips the download when the server reports nothing newer than
    the cached copy, which is what spares re-fetching the 39 MB BoE archive.
    """
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = cache_path(url, prefix)

    if conditional and path.exists():
        head = requests.head(
            url, headers=BROWSER_HEADERS, allow_redirects=True, timeout=HEAD_TIMEOUT
        )
        modified = head.headers.get("Last-Modified")
        cached_at = pd.Timestamp(
            datetime.fromtimestamp(path.stat().st_mtime, tz=UTC)
        )
        if modified is not None and pd.to_datetime(modified, utc=True) <= cached_at:
            return retrieve_from_cache(path)

    try:
        response = requests.get(
            url, headers=BROWSER_HEADERS, timeout=DOWNLOAD_TIMEOUT
        )
        response.raise_for_status()
    except requests.RequestException as error:
        if not path.exists():
            raise
        print(f"Download failed ({error}); using the cached copy of {url}")
        return retrieve_from_cache(path)

    save_to_cache(path, response.content)
    return response.content

In [4]:
def fetch_us_yield(code: str) -> pd.Series:
    """Fetch a daily US Treasury constant-maturity yield from Yahoo Finance."""
    raw = yf.download(code, start=EARLIEST_START, auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        raise ValueError(f"Yahoo Finance returned no data for {code}")
    series = raw["Close"].squeeze().dropna()
    series.index = cast("pd.DatetimeIndex", series.index).to_period("D")
    return series


def fetch_bund_yield(code: str) -> pd.Series:
    """Fetch a daily Bund yield from the Bundesbank time-series API.

    The CSV carries a variable number of metadata rows ahead of the data, and
    publishes non-trading days as a bare full stop.
    """
    url = f"{BBK_BASE}{code}?format=csv&lang=en"
    content = fetch_cached(url, BBK_CACHE_PREFIX)
    lines = content.decode("utf-8-sig").splitlines()
    observations = {
        pd.Period(row[0], freq="D"): float(row[1])
        for row in (line.split(",") for line in lines if ISO_DATE.match(line))
        if row[1] not in (".", "")
    }
    if not observations:
        raise ValueError(f"Bundesbank returned no observations for {code}")
    return pd.Series(observations).sort_index()

In [5]:
def _fetch_mof_csv(path: str) -> pd.DataFrame:
    """Read one MOF JGB yield-curve CSV into a dated frame of per-cent yields.

    These files send Last-Modified and need no headers, so readabs' get_file()
    handles them: fresh when the MOF has published, cached otherwise, and the
    cached copy is used if a download fails. Row 0 is a title line, so the
    header is on row 1, and unquoted tenors are published as a bare hyphen.
    """
    content = get_file(
        MOF_BASE + path, cache_dir=CACHE_DIR, cache_prefix=MOF_CACHE_PREFIX
    )
    text = content.decode(MOF_ENCODING)
    frame = pd.read_csv(io.StringIO(text), skiprows=1, na_values=["-"])
    frame["Date"] = pd.to_datetime(frame["Date"], format="%Y/%m/%d", errors="coerce")
    frame = frame.dropna(subset=["Date"]).set_index("Date")
    return frame.apply(pd.to_numeric, errors="coerce")


@cache
def mof_curve() -> pd.DataFrame:
    """The whole daily JGB curve: the MOF history file, extended by the current
    month. Cached, so every tenor is served from one pair of downloads."""
    history = _fetch_mof_csv(MOF_HISTORY)
    current = _fetch_mof_csv(MOF_CURRENT)
    curve = pd.concat(
        [history, current[~current.index.isin(history.index)]]
    ).sort_index()
    if curve.empty:
        raise ValueError("MOF returned an empty yield curve")
    curve.index = pd.PeriodIndex(curve.index, freq="D")
    return curve


def fetch_japan_yield(code: str) -> pd.Series:
    """Fetch one tenor of the daily JGB yield curve."""
    curve = mof_curve()
    if code not in curve.columns:
        raise ValueError(f"MOF curve has no {code} column: {list(curve.columns)}")
    series = curve[code].dropna()
    if series.empty:
        raise ValueError(f"MOF returned no {code} observations")
    return series

In [6]:
def _boe_spot_curve(archive_bytes: bytes) -> pd.DataFrame:
    """Read the nominal spot curve out of every workbook in a BoE archive.

    The archives hold real and inflation curves too, and the spot sheet was
    renamed from "4. nominal spot curve" to "4. spot curve" from the 2005
    workbook onward, so the sheet is matched on its suffix. Row 3 of the sheet
    holds the maturities in years.

    The current workbook carries a literal "Refresh" row where a date should
    be. Coercing the dates drops that row, but the yield columns stay object
    dtype unless they are coerced too - and an object column plots as a line
    while silently losing its end-point annotation.
    """
    archive = zipfile.ZipFile(io.BytesIO(archive_bytes))
    frames = []
    for name in archive.namelist():
        if "Nominal" not in name:
            continue
        book = pd.ExcelFile(io.BytesIO(archive.read(name)))
        sheets = [
            sheet
            for sheet in book.sheet_names
            if sheet.endswith("spot curve") and "short end" not in sheet
        ]
        if not sheets:
            raise ValueError(f"No nominal spot curve sheet in {name}")
        frame = book.parse(sheets[0], header=3)
        frame = frame.rename(columns={frame.columns[0]: "Date"})
        frame["Date"] = pd.to_datetime(
            frame["Date"], errors="coerce", format="mixed"
        )
        frame = frame.dropna(subset=["Date"]).set_index("Date")
        frames.append(frame.apply(pd.to_numeric, errors="coerce"))
    if not frames:
        raise ValueError("BoE archive held no nominal curve workbook")
    return pd.concat(frames).sort_index()


@cache
def boe_curve() -> pd.DataFrame:
    """The daily fitted nominal gilt curve: the history archive plus this month.

    The history archive is ~39 MB and changes only as years roll over, so it is
    fetched conditionally on the server's Last-Modified.
    """
    history = _boe_spot_curve(
        fetch_cached(BOE_BASE + BOE_HISTORY, BOE_CACHE_PREFIX, conditional=True)
    )
    current = _boe_spot_curve(
        fetch_cached(BOE_BASE + BOE_CURRENT, BOE_CACHE_PREFIX)
    )
    curve = pd.concat(
        [history, current[~current.index.isin(history.index)]]
    ).sort_index()
    curve.index = pd.PeriodIndex(curve.index, freq="D")
    return curve


def fetch_gilt_yield(code: str) -> pd.Series:
    """Fetch one maturity off the BoE nominal gilt curve, in years."""
    curve = boe_curve()
    maturity = float(code)
    if maturity not in curve.columns:
        raise ValueError(f"BoE curve has no {maturity}-year maturity")
    series = curve[maturity].dropna()
    if series.empty:
        raise ValueError(f"BoE returned no {maturity}-year observations")
    return series

In [7]:
def fetch_australia_yield(code: str) -> pd.Series:
    """Fetch a daily RBA Australian Government bond yield, named by its title.

    The RBA split its daily capital market yields in 2013, so the current table
    is extended backwards by the historical one. The series ID is resolved from
    the current table's own metadata rather than written down here, since the
    two tables share IDs but not metadata layouts.
    """
    current, meta = ra.read_rba_table(RBA_CURRENT_TABLE)
    matches = meta[meta["Title"] == code]
    if len(matches) != 1:
        raise ValueError(
            f"RBA {RBA_CURRENT_TABLE} holds {len(matches)} series titled {code!r}"
        )
    series_id = str(matches["Series ID"].iloc[0])
    history, _ = ra.read_rba_table(RBA_HISTORY_TABLE)
    if series_id not in history.columns:
        raise ValueError(f"RBA {RBA_HISTORY_TABLE} has no {series_id} column")
    # keep="last" lets the current table win wherever the two tables overlap
    joined = pd.concat([history[series_id].dropna(), current[series_id].dropna()])
    joined = joined[~joined.index.duplicated(keep="last")].sort_index()
    if joined.empty:
        raise ValueError(f"RBA returned no observations for {code!r}")
    return joined

In [8]:
def fetch_yield(country: str, code: str) -> pd.Series:
    """Fetch one country's yield series from whichever source publishes it.

    The result is forced to a numeric dtype here rather than in each fetcher.
    Several sources hand back object columns - the RBA tables do, and so does a
    BoE workbook carrying its "Refresh" row - and an object column plots as a
    line while mgplot silently skips its end-point annotation. Coercing in one
    place means a new source cannot reintroduce that quietly.
    """
    fetchers = {
        "United States": fetch_us_yield,
        "Japan": fetch_japan_yield,
        "Germany": fetch_bund_yield,
        "United Kingdom": fetch_gilt_yield,
        "Australia": fetch_australia_yield,
    }
    if country not in fetchers:
        raise ValueError(f"No fetcher defined for {country}")
    series = pd.to_numeric(fetchers[country](code), errors="coerce").dropna()
    if series.empty:
        raise ValueError(f"No numeric observations for {country} ({code})")
    return series.rename(country)


def get_yields(name: str, codes: dict[str, str], start: str) -> pd.DataFrame:
    """Assemble one tenor's country series onto a common daily index.

    The markets keep different holidays and their histories begin at different
    dates, so the union index carries gaps in every column; they are left as
    missing rather than filled.
    """
    frame = pd.DataFrame(
        {country: fetch_yield(country, code) for country, code in codes.items()}
    )
    frame = frame[frame.index >= pd.Period(start, freq="D")].dropna(how="all")
    if frame.empty:
        raise ValueError(f"No {name} observations on or after {start}")
    for country in frame.columns:
        column = frame[country].dropna()
        print(
            f"{name} {country}: {column.index[0]} to {column.index[-1]}, "
            f"{len(column)} observations, last {column.iloc[-1]:.2f} per cent"
        )
    return frame


def get_all_yields() -> dict[str, pd.DataFrame]:
    """Fetch every specified tenor."""
    return {
        name: get_yields(name, codes, TENOR_STARTS[name])
        for name, codes in CODES.items()
    }


yields = get_all_yields()

10-year United States: 1986-01-02 to 2026-09-01, 10202 observations, last 4.80 per cent
10-year Japan: 1986-07-05 to 2026-09-01, 9930 observations, last 2.99 per cent
10-year Germany: 1997-08-07 to 2026-09-01, 7378 observations, last 3.32 per cent
10-year United Kingdom: 1986-01-02 to 2026-08-28, 10276 observations, last 5.14 per cent
10-year Australia: 1995-01-03 to 2026-08-26, 7980 observations, last 5.02 per cent


30-year United States: 1999-01-04 to 2026-09-01, 6949 observations, last 5.27 per cent
30-year Japan: 1999-09-02 to 2026-09-01, 6614 observations, last 4.13 per cent
30-year Germany: 2000-08-01 to 2026-09-01, 6626 observations, last 3.79 per cent
30-year United Kingdom: 2016-01-04 to 2026-08-28, 2693 observations, last 5.91 per cent


## Plotting

In [9]:
def join_names(names: list[str]) -> str:
    """Join names as a readable list: "a, b and c"."""
    if len(names) == 1:
        return names[0]
    return f"{', '.join(names[:-1])} and {names[-1]}"


def title_countries(countries: list[str]) -> str:
    """Render a country list for a chart title, shortening the long names."""
    return join_names([SHORT_NAMES.get(country, country) for country in countries])


def data_to_footer(data: pd.DataFrame) -> str:
    """Footer text giving the last observation date of every series.

    These markets publish with different lags, so one "data to" date would
    overstate the freshness of whichever series ended earliest. Countries are
    two-letter coded and those sharing an end date are grouped, most recent
    first.
    """
    ends: dict[pd.Period, list[str]] = {}
    for country in data.columns:
        code = ABBREVIATIONS.get(country, country)
        ends.setdefault(data[country].dropna().index[-1], []).append(code)
    parts = [
        f"{join_names(codes)} {end.strftime('%-d-%b-%Y')}"
        for end, codes in sorted(ends.items(), reverse=True)
    ]
    return f"Data to: {'; '.join(parts)}. "


def plot_yields(name: str, data: pd.DataFrame, note: str | None) -> None:
    """Chart one tenor over the full history and the recent window.

    mgplot type-checks `rheader` as a string, so a tenor with no comparability
    caveat omits the argument rather than passing None.
    """
    header = {"rheader": note} if note is not None else {}
    mg.multi_start(
        data,
        function=mg.line_plot_finalise,
        starts=plot_times,
        title=f"{title_countries(list(data.columns))}: {name} Government Bond Yields",
        ylabel="Per cent per year",
        xlabel=None,
        width=1,
        annotate=True,
        rounding=2,
        legend={"loc": "best", "fontsize": "small"},
        lfooter=data_to_footer(data),
        rfooter=SOURCE,
        show=SHOW,
        **header,
    )


def plot_all_yields(all_data: dict[str, pd.DataFrame]) -> None:
    """Chart every fetched tenor."""
    for name, data in all_data.items():
        plot_yields(name, data, TENOR_NOTES[name])


plot_all_yields(yields)

## Watermark

In [10]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-09-02 11:36:20

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.16.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot  : 0.2.33
pandas  : 3.0.5
pathlib : 1.0.1
re      : 2.2.1
readabs : 0.2.6
requests: 2.34.2
typing  : 3.10.0.0
yfinance: 1.6.0

Watermark: 2.6.0

